# 02_05_Handling Missing Values in `RELATION_DIRECTION`

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")

In [3]:
AGG_BASE = {
    "total_entries" : ("TRAIN_SERV", "count"),
    "missing_values" : ("RELATION_DIRECTION", lambda x: x.isnull().sum())
}

AGG_RELATION_ACTIVITY = {
    **AGG_BASE,
    "relation_start" : ("DATDEP", "min"),
    "relation_end" : ("DATDEP", "max")
}

AGG_MISSING_STATS = {
    **AGG_BASE,
    "missing_start" : ("RELATION_DIRECTION_MISSING_DATE", "min"),
    "missing_end" : ("RELATION_DIRECTION_MISSING_DATE", "max")
}

# Table of Contents

- [5. PUNCTUALITY DATASET - MISSING RELATION DIRECTIONS](#5-punctuality-dataset---missing-relation-directions)

- [5.1 Handling Missing Values in `RELATION_DIRECTION`](#51-handling-missing-values-in-relation_direction)
    - [5.1.1. Handling Missing Values in `RELATION_DIRECTION` for `P`, `L`, `EXTRA`, and `CHARTER` `RELATION`](#511-handling-missing-values-in-relation_direction-for-p-l-extra-and-charter-relation)  
    
    - [5.1.2. Handling Missing Values in `RELATION_DIRECTION` for `IC 41` and `L C3-2` `RELATION`](#512-handling-missing-values-in-relation_direction-for-ic-41-and-l-c3-2-relation)
        - [5.1.2.1. Profiling Missing Directions for `IC 41` Relation](#5121-profiling-missing-directions-for-ic-41-relation)
        - [5.1.2.2. Profiling Missing Directions for `L C3-2` Relation](#5122-profiling-missing-directions-for-l-c3-2-relation)
        - [5.1.2.3. Inferring Directions for `IC 41` and `L C3-2`](#5123-inferring-directions-for-ic-41-and-l-c3-2)

    - [5.1.3. Handling Missing Values in `RELATION_DIRECTION` for `L C2-2` `RELATION`](#513-handling-missing-values-in-relation_direction-for-l-c2-2-relation)
        - [5.1.3.1. Profiling Missing Directions in `L C2-2` Relation](#5131-profiling-missing-directions-in-l-c2-2-relation)
        - [5.1.3.2. Inferring Missing Directions for `L C2-2`](#5132-inferring-missing-directions-for-l-c2-2)

    - [5.1.4. Handling Missing Values in `RELATION_DIRECTION` for `L B5-1` `RELATION`](#514-handling-missing-values-in-relation_direction-for-l-b5-1-relation)
        - [5.1.4.1. Profiling Missing Directions for `L B5-1` Relation](#5141-profiling-missing-directions-for-l-b5-1-relation)
        - [5.1.4.2. Inferring Directions for `L B5-1`](#5142-inferring-directions-for-l-b5-1)

    - [5.1.5. Handling Missing Values in `RELATION_DIRECTION` for `IC 20` `RELATION`](#515-handling-missing-values-in-relation_direction-for-ic-20-relation)
        - [5.1.5.1. Profiling Missing Directions for `IC 20` Relation](#5151-profiling-missing-directions-for-ic-20-relation)
        - [5.1.5.2. Inferring Directions for `IC 20`](#5152-inferring-directions-for-ic-20)

    - [5.1.6. Handling Missing values in `RELATION_DIRECTION` for Remaining `RELATION` Classes](#516-handling-missing-values-in-relation_direction-for-other-relation-classes)

- [5.2. Export to Silver Layer](#52-export-to-silver-layer)

## 5. PUNCTUALITY DATASET - MISSING RELATION DIRECTIONS

In [4]:
punctuality = (
    pd.read_parquet(PATH_INTERMEDIATE / "punctuality_checkpoint_02_04.parquet")
)

## 5.1 Handling Missing Values in `RELATION_DIRECTION`

In the *build_secondary_dimensions* notebook, `RELATION_DIRECTION` column will be extracted from `punctuality` and replaced by a **Surrogate Key** to create a `train_service` dimension, as it is the column with the finest granularity. The `train_service` dimension will include the `TRAIN_SERV`, `RELATION`, and `RELATION_DIRECTION` columns. A derived `RELATION_CATEGORY` column, built from `RELATION`, will also be added to this dimension.
  
1) `TRAIN_SERV` and `RELATION` have respectively 4 and 171 unique values, with **no missing values**.  

2) `RELATION_DIRECTION` has 498 unique values and **2,686,482 missing values**.

3) There are 15 `RELATION` **classes** with **missing values** in `RELATION_DIRECTION`: `IC 20`, `IC 41`, `L 12`, `L 19`, `L 28`, `L 43`, `L B5-1`, `L C2-2`, `L C3-2`, `INT`, `THAL`, `EXTRA`, `P`, `L`, `CHARTER`.

4) Five of them have a missing direction rate of **100%**: `IC 41`, `EXTRA`, `P`, `L`, and `CHARTER`. This is expected for `EXTRA`, `P`, `L`, and `CHARTER`, since these classes are never assigned a direction by definition, as established in notebook *02_04_profiling_and_cleaning_punctuality*.  

5) The four non-IC classes account for **97% of the total missing values** in `RELATION_DIRECTION`.  
**`EXTRA`, `P`, `L`, and `CHARTER` classes** are handled in **Section 5.1.1.**

6) Given that `IC` relations are InterCity train services that are expected to have directions, the 100% missing rate observed for `IC 41` is handled differently from `EXTRA`, `P`, `L`, and `CHARTER` classes. 

**ANALYSIS**

The following table details the **11** `RELATION` **remaining classes**:
* Missing values rates calculated relative to the total entries of each `RELATION` class, not to the total dataset.  

* The number of missing `RELATION_DIRECTION` values per class.  

* The date range and duration over which missing values are spread within the 2024-2025 dataset.  

<br>

| RELATION  | Missing Values Rate   | Missing Values Entries | Duration  | Dates                     |
|-----------|-----------------------|------------------------|-----------|---------------------------|
| IC 41     | 100 %                 | 785                    | 34 days   | 2024-01-01 to 2024-02-04  |
| L C2-2    | 13.8 %                | 4975                   | 98 days   | 2025-09-06 to 2025-12-13  |
| L C3-2    | 7.09 %                | 6912                   | 351 days  | 2024-02-17 to 2025-02-02  |
| IC 20     | 6.75 %                | 55,423                 | 43 days   | 2024-11-01 to 2024-12-14  |
| L B5-1    | 2.42 %                | 10,818                 | 101 days  | 2024-01-02 to 2024-04-12  |
| L 43      | 0.89 %                | 135                    | 398 days  | 2024-01-01 to 2025-02-02  |
| INT       | 0.54 %                | 1455                   | 629 days  | 2024-04-08 to 2025-12-28  |
| L 12      | 0.53 %                | 92                     | 365 days  | 2024-11-11 to 2025-11-11  |
| L 19      | 0.18 %                | 176                    | <1 day    | 2025-11-11                |
| THAL      | 0.008 %               | 20                     | <1 day    | 2025-12-21                |
| L 28      | 0.003 %               | 3                      | <1 day    | 2024-09-16                |

* With a **100 % missing values rate** and **34 days of activity**, the **IC 41** train service appears as a **legacy entry**. According to the *SNCB Plan de transport 2023-2026* documentation, this relation was officially reclassified as `S63`. The 785 recorded instances between 1 January and 4 February 2024 reflect a transitional period during which the old "IC" label had not yet been updated in the dataset.  
**L C3-2** appears to have served as a temporary remplacement for `IC 41` prior to the permanent `S63` new label, serving the same route with two additional stops whithin the Charleroi urban area.  
**`IC 41` and `L C3-2` missing directions** are investigated and handled in **Section 5.1.2.**.

* **L C2-2**  exhibits the second highest missing value rate: **13.8 %**. Like `L B5-1`, `L C3-2`, and `IC 41`, this train service was relabeled as `S62-2` during the `punctuality` dataset period. But this label change alone cannot explain the missing directions: they span **98 days**, which is **too long a period to result from a simple relabeling transition**. Engineering works are equally unlikely, as the missing direction dates do not align with those of `L C3-2`, which serves the same infrastructure line — concurrent works would be expected to affect both relations within the same time window. Their origin therefore remains unclear. However, since L C2-2 has only two possible direction values, it was possible to reassign the correct directions using the train numbers serving this relation.  
**`L C2-2` missing directions** are investigated and handled in the **Section 5.1.3.**.

* **L B5-1** has a missing value rate of **2.42%**. Part of the `L B5-1` train service was relabeled as `S5-1` in 14 December 2025. However, the missing `RELATION_DIRECTION` values are all concentrated between 2 January 2024 and 12 April 2024 - **well before the relabeling**. Therefore, this label change cannot explain these direction losses. Furthermore, other entries for the same relation and period do carry `RELATION_DIRECTION` values. As for `IC 20`, a desynchronization between real-time tracking (*ARAMIS system*) and platform route planing (*OP1 platform*) linked to *Brussels RER* engineering works may explain these missing directions, though the evidences are insufficient to confirm this.  
**`L B5-1` missing directions** are investigated and handled in the **Section 5.1.4.**.

* **IC 20** has a high missing value rate of **6.75%**, concentrated in **a six-week window** despite the relation spanning the full 2024-2025 period. This time window may align with engineering works on the *Brussels RER (Réseau Express Régional)*, which may have explain a desynchronization between real-time tracking (managed by the *ARAMIS system*) and platform route planning (managed by the *OP1 platform*), resulting in direction data loss. Nevertheless, insuffisiant evidence exists to confirm the hypothesis.  
**`IC 20` missing directions** are investigated and handled in the **Section 5.1.5.**.

* **L 19**, **THAL**, and **L 28** each have a very low missing values rate (< 0.2%), each focused on one day only. This is likely caused by an isolated server error - or, in the case of the Thalys train, a cross-operator IT or database maintenance event on that day.

* **INT**, **L 12**, and **L 43** are train services that cross national borders: **L 12** between Antwerpen-Centraal and the Dutch border, **L 43** between Liège and the Luxembourg border, and **INT** covering various international routes served by multiple operators. Their low missing value rate (< 1%) spread over long periods of time (between 365 and 629 days) are consistant with occasional data loss during handover between national operators at the border. In addition, train numbers can change when they cross the border, which is likely to increase the risk of occasional data loss.

* That said, without stronger evidence, these explanations for **L 19**, **THAL**, **L 28**, **INT**, **L 12**, and **L 43** remain hypotheses.  
**The missing directions of these classes** are investigated and handled in the **Section 5.1.6.**.



**DECISION**  

* Before applying any change, a derived binary column `DIRECTION_IS_INFERRED` is added to the dataset to **flag** the original missing values in `RELATION_DIRECTION`, and another derived column `RELATION_DIRECTION_MISSING_DATE` is added to preserve the occurrence dates of these missing directions for subsequent temporal analysis.

* For `RELATION` classes that never have an operational direction - `EXTRA`, `L`, `P`, and `CHARTER` -, the `RELATION` value is copied into `RELATION_DIRECTION` to avoid NULL values in the future SQL data warehouse and ensure the integrity of the `train_service` dimension.

* **The correct `IC 41` missing directions** are recovered from the `S63` `RELATION_DIRECTION` values and the train numbers (`TRAIN_NO` unique values) serving both relations.

* Likewise, **the correct `L C2-2` missing directions** are recovered from the `S62-2` `RELATION_DIRECTION` values and the train numbers serving both relations.

* **For `L B5-1` and `L C3-2` missing directions**, we can infer the general direction from `TRAIN_NO` unique values - towards **MECHELEN** for `L B5-1`, and towards **CHARLEROI** for `L C3-2` - but we cannot infer the exact downstream terminus. Consequently, we replace the missing values by:
    * `L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES / MAUBEUGE`
    * `L C3-2: ERQUELINNES / MAUBEUGE -> CHARLEROI-CENTRAL`
    * `L B5-1: MECHELEN -> HALLE / EDINGEN`
    * `L B5-1: HALLE / EDINGEN -> MECHELEN`

* `IC 20` is a major train service with six distinct non-null `RELATION_DIRECTION` values. As a result, we cannot infer the general direction of this service, and **`IC 20` missing directions** are categorized as `"IC 20 - Direction Unknown_Works impact"`.

* For **L 19**, **THAL**, **L 28**, **INT**, **L 12**, and **L 43**, the missing values rate and their distribution over time suggests either IT incidents or data loss during borders crossing and national operator handovers, but without stronger evidence, these remain hypotheses. Nevertheless, since the missing directions represent less than **1%** of the entries for these relations and less than **0.01%** of the total `punctuality` entries, they are replaced by the relation name followed by the *Direction Unknown* suffix (e.g. `L 43 - Direction Unknown`).


In [5]:
#Adding derived binary column to flag the original missing directions
punctuality["DIRECTION_IS_INFERRED"] = punctuality["RELATION_DIRECTION"].isnull().astype("Int64")

In [6]:
#Verification of the derived flag column
print(punctuality["DIRECTION_IS_INFERRED"].sum() == punctuality["RELATION_DIRECTION"].isnull().sum())

True


In [7]:
#Adding derived datetime64 column to capture occurrence dates of missing values
punctuality["RELATION_DIRECTION_MISSING_DATE"] = (
     punctuality["DATDEP"] .where(punctuality["RELATION_DIRECTION"].isnull())
)

In [8]:
# Verification of the derived datetime64 column
print(
    punctuality["RELATION_DIRECTION_MISSING_DATE"].notna().sum() == punctuality["DIRECTION_IS_INFERRED"].sum()
)

True


In [9]:
punctuality[["TRAIN_SERV", "RELATION", "RELATION_DIRECTION"]].nunique()

TRAIN_SERV              4
RELATION              171
RELATION_DIRECTION    498
dtype: int64

In [10]:
punctuality[["TRAIN_SERV", "RELATION", "RELATION_DIRECTION"]].isnull().sum()

TRAIN_SERV                  0
RELATION                    0
RELATION_DIRECTION    2686482
dtype: int64

In [11]:
punctuality_without_direction = (
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION_DIRECTION"].isnull()]
                               )

relations_to_check = punctuality_without_direction["RELATION"].drop_duplicates().reset_index(drop=True)

In [12]:
punctuality_to_check = punctuality[["RELATION", 
                                    "RELATION_DIRECTION", 
                                    "TRAIN_SERV", 
                                    "DATDEP", 
                                    "LINE_NO_ARR",
                                    "TRAIN_NO",
                                    "PTCAR_NO",
                                    "PTCAR_LG_NM_NL",
                                    "DIRECTION_IS_INFERRED",
                                    "RELATION_DIRECTION_MISSING_DATE"]]

In [13]:
print("------------------------------RELATIONS WITH MISSING DIRECTIONS------------------------------")
relations_with_missing_directions = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(relations_to_check)]
    .groupby("RELATION").agg(
        **AGG_RELATION_ACTIVITY
    )
)

relations_with_missing_directions["pct_of_dataset"] = (
    relations_with_missing_directions["total_entries"] / punctuality.shape[0] * 100
    ).round(3)

relations_with_missing_directions["relation_active_days"] = (
    (relations_with_missing_directions["relation_end"] - relations_with_missing_directions["relation_start"])
)

relations_with_missing_directions = (
    relations_with_missing_directions[["total_entries", "missing_values", "pct_of_dataset",
                                       "relation_start", "relation_end", "relation_active_days"]]
)

relations_with_missing_directions.sort_values("pct_of_dataset", ascending=False)


------------------------------RELATIONS WITH MISSING DIRECTIONS------------------------------


,total_entries,missing_values,pct_of_dataset,relation_start,relation_end,relation_active_days
RELATION,,,,,,
P,1770742,1770742,3.888,2024-01-02,2025-12-31,729 days
IC 20,821563,55423,1.804,2024-01-01,2025-12-31,730 days
EXTRA,795494,795494,1.747,2024-01-01,2025-12-31,730 days
L B5-1,446585,10818,0.981,2024-01-01,2025-12-13,712 days
INT,271493,1455,0.596,2024-01-01,2025-12-31,730 days
THAL,258694,20,0.568,2024-01-01,2025-12-21,720 days
L C3-2,97546,6912,0.214,2024-01-02,2025-12-13,711 days
L 19,96115,176,0.211,2024-01-02,2025-12-31,729 days
L 28,91835,3,0.202,2024-01-01,2025-12-28,727 days


* `IC 41` was only active for **34 days** in 2024-2025 and accounts for less than **0.01%** of `punctuality` entries.  

* `L B5-1`, `L C2-2`, and `L C3-2` train services **all end on 13 December 2025**. According to the *SNCB Plan de transport 2023-2026*, this date corresponds to the beginning of **the third phase of the SNCB Plan 2023-2026** and may indicate that these relations have changed or have been relabeled after this date.  

* `CHARTER` has a low number of entries - 6874 out of 45.5 Millions entries - and was active for a duration of 622 days. Given that `CHARTER` trains are, by definition, added to the railway network for specific events, as explained in the *02_04_profiling_and_cleaning_punctuality* notebook, this was expected.  

* `L` train service only accounts for **0.072 %** of `punctuality`, starts on 3 February 2024 and ends on 12 December 2025. This low number of entries and incomplete span of time (678 out of 730 days) will be investigated in the *03_02_build_dimension_train_service* notebook.  

* `THAL` train service didn't run the last ten days of 2025 for an unknown reason. If needed, this also will be investigated in the *03_02_build_dimension_train_service* notebook.

In [14]:
print("---------------------------------MISSING DIRECTIONS ANALYSIS---------------------------------")
missing_values_only = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(relations_to_check)]
    .groupby("RELATION").agg(
        **AGG_MISSING_STATS
    )
)

missing_values_only["total_entries"] = relations_with_missing_directions["total_entries"]

missing_values_only["pct_of_missing"] = (
    missing_values_only["missing_values"] / missing_values_only["total_entries"]  * 100
    ).round(3)

missing_values_only["missing_span_days"] = (
    (missing_values_only["missing_end"] - missing_values_only["missing_start"])
)

missing_values_only = (
    missing_values_only[["total_entries", "missing_values", "pct_of_missing",
                        "missing_start", "missing_end", "missing_span_days"]]
)

missing_values_only.sort_values("pct_of_missing", ascending=False)

---------------------------------MISSING DIRECTIONS ANALYSIS---------------------------------


,total_entries,missing_values,pct_of_missing,missing_start,missing_end,missing_span_days
RELATION,,,,,,
CHARTER,6874,6874,100.0,2024-03-01,2025-11-13,622 days
EXTRA,795494,795494,100.0,2024-01-01,2025-12-31,730 days
IC 41,785,785,100.0,2024-01-01,2024-02-04,34 days
L,32758,32758,100.0,2024-02-03,2025-12-12,678 days
P,1770742,1770742,100.0,2024-01-02,2025-12-31,729 days
L C2-2,34746,4795,13.8,2025-09-06,2025-12-13,98 days
L C3-2,97546,6912,7.086,2024-02-17,2025-02-02,351 days
IC 20,821563,55423,6.746,2024-11-01,2024-12-14,43 days
L B5-1,446585,10818,2.422,2024-01-02,2024-04-12,101 days


* Missing `RELATION_DIRECTION` values in `CHARTER`, `EXTRA`, `L`, `P`, and `IC 41` account for **100%** of their total entries.

* Missing `RELATION_DIRECTION` values in `L 43`, `INT`, `L 12`, `L 19`, `THAL`, and `L 28` account for **less than 1%** of their entries. In addition, the missing direction values in `L 19`, `THAL`, and `L 28` are all **concentrated within a single day**.

* The four remaining `RELATION` classes - `L C2-2`, `L C3-2`, `IC 20`, and `L B5-1` - have a missing direction rate ranging from **~2.4 %** to **13.8 %**.

In [15]:
relations_never_with_direction = ["P", "L", "EXTRA", "CHARTER"]

punctuality_never_direction = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(relations_never_with_direction)]
                                       )

Given that it is an InterCity train service that should have directions, `IC 41` is excluded from the `relations_never_with_direction` list above.

In [16]:
no_missing_values_total = punctuality_without_direction.shape[0]
no_missing_values_never_direction = punctuality_never_direction.shape[0]
no_missing_values_still_to_check = no_missing_values_total - no_missing_values_never_direction
ratio_missing_values_still_to_check = round(no_missing_values_still_to_check / no_missing_values_total * 100, 2)

print(f"Missing values in RELATION_DIRECTION: {punctuality_without_direction.shape[0]}")
print(f"Missing values in RELATION_DIRECTION for EXTRA, P, L, and CHARTER classes: "
      f"{punctuality_never_direction.shape[0]}")
print(f"There remain {no_missing_values_total - no_missing_values_never_direction} missing values to investigate.")
print(f"This number represents precisely {ratio_missing_values_still_to_check}% of the total missing values "
      f"in the RELATION_DIRECTION column.")

Missing values in RELATION_DIRECTION: 2686482
Missing values in RELATION_DIRECTION for EXTRA, P, L, and CHARTER classes: 2605868
There remain 80614 missing values to investigate.
This number represents precisely 3.0% of the total missing values in the RELATION_DIRECTION column.


In [17]:
relations_still_to_check = (
    relations_to_check.loc[~relations_to_check.isin(relations_never_with_direction)]
    )
relations_still_to_check.reset_index(drop=True).to_list()

['IC 41',
 'L 43',
 'L B5-1',
 'L C3-2',
 'INT',
 'L 28',
 'IC 20',
 'L 12',
 'L C2-2',
 'L 19',
 'THAL']

In [18]:
print(f"These {no_missing_values_still_to_check} missing values are " 
      f"distributed across {len(relations_still_to_check)} relations other than EXTRA, P, L, and CHARTER classes.")


These 80614 missing values are distributed across 11 relations other than EXTRA, P, L, and CHARTER classes.


In [19]:
punctuality_directions_still_missing = punctuality_to_check.loc[
    punctuality["RELATION"].isin(relations_still_to_check) &
    punctuality["RELATION_DIRECTION"].isnull()
]

In [20]:
lines_without_relation_directions =  (
    punctuality_directions_still_missing["LINE_NO_ARR"].dropna().drop_duplicates().reset_index(drop=True)
)
train_no_without_relation_directions = (
    punctuality_directions_still_missing["TRAIN_NO"].dropna().drop_duplicates().reset_index(drop=True)
)
pt_car_without_relation_directions = (
    punctuality_directions_still_missing["PTCAR_NO"].dropna().drop_duplicates().reset_index(drop=True)
)
print(f"{len(lines_without_relation_directions)} infrastructure lines are served by "
      f"relations with missing directions.")
print(f"{len(train_no_without_relation_directions)} train numbers are served by "
      f"relations with missing directions.")
print(f"{len(pt_car_without_relation_directions)} stations or train stops are served by " 
      f"relations with missing directions.")

52 infrastructure lines are served by relations with missing directions.
187 train numbers are served by relations with missing directions.
216 stations or train stops are served by relations with missing directions.


### 5.1.1. Handling Missing Values in `RELATION_DIRECTION` for `P`, `L`, `EXTRA`, and `CHARTER` `RELATION`

The `RELATION` `P`, `L`, `EXTRA`, and `CHARTER` classes are assigned to the corresponding missing values in `RELATION_DIRECTION`, since they never have operational direction by definition.

In [21]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"]
                                                        .isin(relations_never_with_direction)]
                                                    .drop_duplicates().reset_index(drop=True)
    )

,RELATION,RELATION_DIRECTION
0,EXTRA,<NA>
1,P,<NA>
2,L,<NA>
3,CHARTER,<NA>


In [22]:
always_missing = punctuality["RELATION"].isin(["P", "L", "EXTRA", "CHARTER"])

punctuality.loc[
    always_missing & punctuality["RELATION_DIRECTION"].isnull(), "RELATION_DIRECTION"
] = punctuality.loc[
    always_missing & punctuality["RELATION_DIRECTION"].isnull(), "RELATION"
]

In [23]:
# Verification
(
    punctuality.loc[(punctuality["RELATION"].isin(relations_never_with_direction)) 
                    & (punctuality["RELATION_DIRECTION"].isnull())]
)

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [24]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"]
                                                        .isin(relations_never_with_direction)]
                                                    .drop_duplicates().reset_index(drop=True)
    )

,RELATION,RELATION_DIRECTION
0,EXTRA,EXTRA
1,P,P
2,L,L
3,CHARTER,CHARTER


This section resolves **97%** of the missing direction values.

### 5.1.2. Handling Missing Values in `RELATION_DIRECTION` for `IC 41` and `L C3-2` `RELATION`

### 5.1.2.1. Profiling Missing Directions for `IC 41` Relation

As established in Section 5.1.:
* `IC 41` was active **34 days** only, from 1 January 2024 until 4 February 2024.
* Its missing direction rate is **100%**.

As shown below:  
* `IC 41` is a small train service that serves **12 stations** and has only **6 train numbers**. 

* With a **100%** missing direction rate, `IC 41` has obviously **0** `RELATION_DIRECTION` values.

In [25]:
train_service_ic41 = ["IC 41"]

punctuality_ic41 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(train_service_ic41)]

print(f"IC 41 total entries: {punctuality_ic41.shape[0]}")
print(f"The IC 41 entries represent {round(punctuality_ic41.shape[0] / punctuality.shape[0] * 100, 4)}% "
      f"of the total entries of the punctuality dataset.")
print(f" ")
print(f"TRAIN_NO: {len(punctuality_ic41["TRAIN_NO"].dropna().drop_duplicates())}")
print(f"RELATION_DIRECTION: {len(punctuality_ic41["RELATION_DIRECTION"].dropna().drop_duplicates())}")
print(f"Number of Stations: {len(punctuality_ic41["PTCAR_LG_NM_NL"].dropna().drop_duplicates())}")

IC 41 total entries: 785
The IC 41 entries represent 0.0017% of the total entries of the punctuality dataset.
 
TRAIN_NO: 6
RELATION_DIRECTION: 0
Number of Stations: 12


In [26]:
train_no_ic41 = punctuality_ic41["TRAIN_NO"].drop_duplicates().to_list()
print(f"TRAIN_NO unique values associated to IC 41: {train_no_ic41}.")

TRAIN_NO unique values associated to IC 41: [19831, 19829, 19828, 19827, 19826, 19825].


In [27]:
station_names_ic41 = punctuality_ic41["PTCAR_LG_NM_NL"].drop_duplicates().to_list()
print("---Station names served by IC 41---")
station_names_ic41

---Station names served by IC 41---


['ERQUELINNES',
 'ERQUELINNES-VILLAGE',
 'SOLRE-SUR-SAMBRE',
 'LABUISSIERE',
 'FONTAINE-VALMONT',
 'LOBBES-GARAGE',
 'LOBBES',
 'THUIN',
 'HOURPES',
 'LANDELIES',
 'MARCHIENNE-ZONE',
 'CHARLEROI-CENTRAL']

In [28]:
punctuality_train_no_relaton_direction_ic41 = (
    punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[punctuality_to_check["TRAIN_NO"].isin(train_no_ic41)]
    .drop_duplicates().sort_values("TRAIN_NO").reset_index(drop=True)
    )

punctuality_train_no_relaton_direction_ic41.head(10)

,TRAIN_NO,RELATION,RELATION_DIRECTION
0,19825,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
1,19825,IC 41,<NA>
2,19825,L C3-2,<NA>
3,19825,L C3-2,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
4,19825,S63,S63: ERQUELINNES -> CHARLEROI-CENTRAL
5,19826,S63,S63: CHARLEROI-CENTRAL -> ERQUELINNES
6,19826,L C3-2,<NA>
7,19826,IC 41,<NA>
8,19826,L C3-2,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE
9,19827,L C3-2,<NA>


* Unique `TRAIN_NO` values of `IC 41` also serve `L C3-2` and `S63` train services.

* `L C3-2` and `S63` `RELATION` classes have non-null values in the `RELATION_DIRECTION` column.

In [29]:
relations_same_train_no_ic41 = (
    punctuality_train_no_relaton_direction_ic41["RELATION"].drop_duplicates().sort_values().to_list()
)

In [30]:
punctuality_ic41_lc32_s63 = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(relations_same_train_no_ic41)]
)

alternative_relation_dates_ic41 = (
    punctuality_ic41_lc32_s63.groupby("RELATION").agg(**AGG_RELATION_ACTIVITY).sort_values("relation_start")
)

alternative_relation_dates_ic41["relation_duration"] = (
    alternative_relation_dates_ic41["relation_end"] - alternative_relation_dates_ic41["relation_start"]
    )

alternative_relation_missing_spreads_ic41 = (
    punctuality_ic41_lc32_s63.groupby("RELATION").agg(**AGG_MISSING_STATS).sort_values("missing_start")
)

alternative_relation_missing_spreads_ic41["missing_spread"] = (
    alternative_relation_missing_spreads_ic41["missing_end"] 
    - alternative_relation_missing_spreads_ic41["missing_start"]
)

print("----------------ALTERNATIVE IC 41 RELATIONS DATES--------------")
display(alternative_relation_dates_ic41)
print(" ")
print("---------ALTERNATIVE IC 41 MISSING DIRECTIONS SPREAD-----------")
alternative_relation_missing_spreads_ic41


----------------ALTERNATIVE IC 41 RELATIONS DATES--------------


,total_entries,missing_values,relation_start,relation_end,relation_duration
RELATION,,,,,
IC 41,785,785,2024-01-01,2024-02-04,34 days
L C3-2,97546,6912,2024-01-02,2025-12-13,711 days
S63,4704,0,2025-12-15,2025-12-31,16 days


 
---------ALTERNATIVE IC 41 MISSING DIRECTIONS SPREAD-----------


,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
IC 41,785,785,2024-01-01,2024-02-04,34 days
L C3-2,97546,6912,2024-02-17,2025-02-02,351 days
S63,4704,0,NaT,NaT,NaT


* `S63` starts two days after `L C3-2` ends.

* `IC 41` and `L C3-2` coexist in the dataset during the first 34 days of 2024. During this period, `IC 41` has a **100%** missing `RELATION_DIRECTION` value rate. This `RELATION` value appears to be a **legacy** from earlier years.

* `L C3-2` is investigated in Section 5.1.2.2.

In [31]:
station_names_s63 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality_to_check["RELATION"] == "S63"]
    .drop_duplicates().to_list()
)

In [32]:
print(station_names_ic41 == station_names_s63[::-1])

True


`IC 41` and `S63` appear to be the same train service, as they serve exactly the **same stations in the same order along the same railway line**.

In [33]:
train_no_s63 = (punctuality_to_check["TRAIN_NO"]
                .loc[punctuality_to_check["RELATION"] == "S63"]
                .drop_duplicates().reset_index(drop=True)
)

As shown in the output below, the `TRAIN_NO` unique values can be used **to infer** the right `IC 41` `RELATION_DIRECTION` values.

In [34]:
ic41_s63 = ["IC 41", "S63"]

(
    punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[(punctuality_to_check["TRAIN_NO"].isin(train_no_ic41))
         & (punctuality_to_check["TRAIN_NO"].isin(train_no_s63))
         & (punctuality_to_check["RELATION"].isin(ic41_s63))]
    .drop_duplicates().reset_index(drop=True).sort_values("TRAIN_NO")
)

,TRAIN_NO,RELATION,RELATION_DIRECTION
5,19825,IC 41,<NA>
9,19825,S63,S63: ERQUELINNES -> CHARLEROI-CENTRAL
8,19826,S63,S63: CHARLEROI-CENTRAL -> ERQUELINNES
4,19826,IC 41,<NA>
3,19827,IC 41,<NA>
6,19827,S63,S63: ERQUELINNES -> CHARLEROI-CENTRAL
2,19828,IC 41,<NA>
7,19828,S63,S63: CHARLEROI-CENTRAL -> ERQUELINNES
11,19829,S63,S63: ERQUELINNES -> CHARLEROI-CENTRAL
1,19829,IC 41,<NA>


**Note**: The next four cells investigate the one-day gap between the end of `L C3-2` (13 December 2025) and the start of `S63` (15 December 2025). Their results are summarized in the conclusion at the end of this section.

In [35]:
station_names_without_network_hub_ic41 = station_names_ic41

station_names_without_network_hub_ic41.remove("CHARLEROI-CENTRAL")

In [36]:
relations_with_same_stations_ic41 = (
    punctuality_to_check["RELATION"]
    .loc[punctuality_to_check["PTCAR_LG_NM_NL"].isin(station_names_without_network_hub_ic41)]
    .drop_duplicates().reset_index(drop=True)
    )
relations_with_same_stations_ic41

0     IC 41
1    L C3-1
2    L C3-2
3         P
4     EXTRA
5       INT
6       S63
Name: RELATION, dtype: string

In [37]:
mystery_14_dec = (
    punctuality_to_check.loc[(punctuality_to_check["RELATION"].isin(relations_with_same_stations_ic41))
                             & (punctuality_to_check["DATDEP"] == "2025-12-14")]
                             
)
mystery_14_dec[["DATDEP", "RELATION"]].value_counts()

DATDEP      RELATION
2025-12-14  EXTRA       1009
            INT          883
            P            363
Name: count, dtype: int64

In [38]:
alternative_relations = ["EXTRA","P"]

alternative_ic41_relations_at_14_12_25 = (
    mystery_14_dec.loc[mystery_14_dec["RELATION"].isin(alternative_relations)]
)

print(f"EXTRA and P train services on this line at 14 December 2025: "
     f"{round(alternative_ic41_relations_at_14_12_25.shape[0] / len(station_names_ic41), 2)}")

EXTRA and P train services on this line at 14 December 2025: 124.73


As shown in the section above:

* The `IC 41` `TRAIN_NO` unique values are associated with two other train services: `L C3-2` and `S63`.

* `IC 41` and `S63` are the same train service, as they serve **the same stations in the same order**.

* `S63` has no missing values.

* `L C3-2` has missing directions and will be investigated in the next section.

* Given their starting and ending dates, these train services may have been replacements for one another. From 1 January to 4 February 2024, `IC 41` has only missing directions and appears to be a **legacy relation** from previous years out of the scope of `punctuality`. `L C3-2` starts from 2 January 2024 until 13 December 2025 while `S63` starts two days later, from 15 December 2025 until the end of the `punctuality` time scope. 

* Even though `S63` accounts for more `TRAIN_NO` unique values than `IC 41`, the `TRAIN_NO` values appearing in both relations always follow the **same directions**:  
    * `ERQUELINNES -> CHARLEROI-CENTRAL`  
    * `CHARLEROI-CENTRAL -> ERQUELINNES`  

* Therefore, we will use these `TRAIN_NO` values to assign the right direction to the `IC 41` `RELATION_DIRECTION` missing values in **Section 5.1.2.3.**.

**Note:** The one-day gap between `L C3-2` and `S63` on **14 December 2025** coincides with the official launch of the third phase of the *SNCB Transport Plan 2023-2026*. On that date, approximately **114** `EXTRA` and `P` trains served the stations normally served by `L C3-2` and `S63` - suggesting that an internal IT delay postponed the registration of `S63` by 24 hours, and that the `EXTRA` and `P` classes were used instead. However, this one-day gap does not affect data quality, as neither `DATDEP` nor `RELATION` contains missing values.

### 5.1.2.2. Profiling Missing Directions for `L C3-2` Relation

As established in Section 5.1.:  
* `L C3-2` was an active relation for **351 days**, from 17 February 2024 to 2 February 2025.  

* Its missing direction rate is **7.09%**.

As shown below:
* `L C3-2` serves 14 stations (two more than `IC 41`), but has a larger number of `TRAIN_NO` unique values - **17** against **6**.  

* This train service has four non-null unique `RELATION_DIRECTION` values: 
    * L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
    * L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
    * L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
    * L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE

* And therefore three terminuses: `MAUBEUGE` and `ERQUELINNES` downstream, and `CHARLEROI-CENTRAL` upstream.

In [39]:
train_service_lc32 = ["L C3-2"]

punctuality_lc32 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(train_service_lc32)]


print(f"L C3-2 total entries: {punctuality_lc32.shape[0]}")
print(f"The L C3-2 entries represent {round(punctuality_lc32.shape[0] / punctuality.shape[0] * 100, 3)}% "
      f"entries of the dataset punctuality.")
print(" ")
print(f"TRAIN_NO: {len(punctuality_lc32["TRAIN_NO"].dropna().drop_duplicates())}")
print(f"RELATION_DIRECTION: {len(punctuality_lc32["RELATION_DIRECTION"].dropna().drop_duplicates())}")
print(f"Number of Stations: {len(punctuality_lc32["PTCAR_LG_NM_NL"].dropna().drop_duplicates())}")

L C3-2 total entries: 97546
The L C3-2 entries represent 0.214% entries of the dataset punctuality.
 
TRAIN_NO: 17
RELATION_DIRECTION: 4
Number of Stations: 14


In [40]:
punctuality_lc32[["RELATION_DIRECTION"]].drop_duplicates().reset_index(drop=True)

,RELATION_DIRECTION
0,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
1,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
2,<NA>
3,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
4,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE


In [41]:
train_no_lc32 = punctuality_lc32["TRAIN_NO"].drop_duplicates().to_list()

In [42]:
station_names_lc32 = punctuality_lc32["PTCAR_LG_NM_NL"].drop_duplicates().to_list()

In [43]:
print(station_names_lc32 == station_names_s63)
remaining_stations_lc32 = list(set(station_names_lc32) - set(station_names_s63))
remaining_stations_lc32

False


['MARCINELLE', 'CHARLEROI-FAISCEAU A']

In [44]:
stations = pd.read_parquet(PATH_INTERMEDIATE / "stations_cleaned.parquet")
stations[["longnamefrench", "class_en"]].loc[stations["longnamefrench"].isin(remaining_stations_lc32)]

,longnamefrench,class_en
566,CHARLEROI-FAISCEAU A,Station
722,MARCINELLE,Station


* `L C3-2` serves the same stations as `S63` or `IC 41`, plus two additional `Station` values: `MARCINELLE` and `CHARLEROI-FAISCEAU A`. These two stations are part of the **Charleroi network hub**, as Marcinelle is a district of the municipality of Charleroi.

* Overall, `L C3-2` is nearly the same train service as `S63` and `IC 41`, with only two extra stops within the Charleroi urban area.

In [45]:
punctuality_lc32_relation_direction_train_no = (
    punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[(punctuality_to_check["TRAIN_NO"].isin(train_no_lc32))
         & (punctuality_to_check["RELATION"] == "L C3-2")]
    .drop_duplicates().sort_values("TRAIN_NO").reset_index(drop=True)
    )

punctuality_lc32_relation_direction_train_no.head(10)

,TRAIN_NO,RELATION,RELATION_DIRECTION
0,19825,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
1,19825,L C3-2,<NA>
2,19825,L C3-2,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
3,19826,L C3-2,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE
4,19826,L C3-2,<NA>
5,19827,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
6,19827,L C3-2,<NA>
7,19827,L C3-2,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
8,19828,L C3-2,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE
9,19828,L C3-2,<NA>


* The same `TRAIN_NO` unique values always follow the same general orientation - **from `CHARLEROI-CENTRAL`** or **to `CHARLEROI-CENTRAL`**. 

* However, they have two possible downstream endpoints: `ERQUELINNES` or `MAUBEUGE`.

* Therefore, we will use these `TRAIN_NO` values to assign one of the two following general directions to the `L C3-2` `RELATION_DIRECTION` missing values in the **Section 5.1.2.3.** :
    * `L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES / MAUBEUGE`
    * `L C3-2: ERQUELINNES / MAUBEUGE -> CHARLEROI-CENTRAL`

### 5.1.2.3. Inferring Directions for `IC 41` and `L C3-2`

As a reminder, before inferring the missing directions for **`IC 41`**, here is the current state of its `RELATION_DIRECTION` values.

In [46]:
(
        punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
        .loc[punctuality_to_check["RELATION"] == "IC 41"]
        .drop_duplicates().reset_index(drop=True).sort_values("TRAIN_NO")
    )

,TRAIN_NO,RELATION,RELATION_DIRECTION
5,19825,IC 41,<NA>
4,19826,IC 41,<NA>
3,19827,IC 41,<NA>
2,19828,IC 41,<NA>
1,19829,IC 41,<NA>
0,19831,IC 41,<NA>


In [47]:
ic41_s63 = ["IC 41", "S63"]

punctuality_s63_ic41 = (
    punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[punctuality_to_check["RELATION"].isin(ic41_s63)]
    .drop_duplicates().reset_index(drop=True)
)

In [48]:
to_charleroi = "IC 41:ERQUELINNES -> CHARLEROI-CENTRAL"
to_erquelinnes = "IC 41:CHARLEROI-CENTRAL -> ERQUELINNES"

to_charleroi_trains = (
    punctuality_s63_ic41["TRAIN_NO"]
    .loc[punctuality_s63_ic41["RELATION_DIRECTION"] == "S63: ERQUELINNES -> CHARLEROI-CENTRAL"]
)
to_erquelinnes_trains = (
    punctuality_s63_ic41["TRAIN_NO"]
    .loc[punctuality_s63_ic41["RELATION_DIRECTION"] == "S63: CHARLEROI-CENTRAL -> ERQUELINNES"]
)

mask_ic41_to_erquelinnes = (
    (punctuality["RELATION"] == "IC 41") & 
    (punctuality["RELATION_DIRECTION"].isnull()) & 
    (punctuality["TRAIN_NO"].isin(to_erquelinnes_trains))
)

mask_ic41_to_charleroi = (
    (punctuality["RELATION"] == "IC 41") & 
    (punctuality["RELATION_DIRECTION"].isnull()) & 
    (punctuality["TRAIN_NO"].isin(to_charleroi_trains))
)

punctuality.loc[mask_ic41_to_erquelinnes, "RELATION_DIRECTION"] = to_erquelinnes
punctuality.loc[mask_ic41_to_charleroi, "RELATION_DIRECTION"] = to_charleroi

In [49]:
# Verification
punctuality.loc[(punctuality["RELATION"] == "IC 41") & (punctuality["RELATION_DIRECTION"].isnull())]

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [50]:
(
    punctuality[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "IC 41"]
    .drop_duplicates().reset_index(drop=True)
)

,TRAIN_NO,RELATION,RELATION_DIRECTION
0,19831,IC 41,IC 41:ERQUELINNES -> CHARLEROI-CENTRAL
1,19829,IC 41,IC 41:ERQUELINNES -> CHARLEROI-CENTRAL
2,19828,IC 41,IC 41:CHARLEROI-CENTRAL -> ERQUELINNES
3,19827,IC 41,IC 41:ERQUELINNES -> CHARLEROI-CENTRAL
4,19826,IC 41,IC 41:CHARLEROI-CENTRAL -> ERQUELINNES
5,19825,IC 41,IC 41:ERQUELINNES -> CHARLEROI-CENTRAL


As a reminder, before inferring the missing directions for `L C3-2`, here is the current state of its `RELATION_DIRECTION` values.

In [51]:
(
    punctuality_lc32[["TRAIN_NO", "RELATION_DIRECTION"]]
    .drop_duplicates().reset_index(drop=True).sort_values("TRAIN_NO")
).head(10)

,TRAIN_NO,RELATION_DIRECTION
3,19825,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
16,19825,<NA>
29,19825,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
28,19826,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE
18,19826,<NA>
5,19827,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
15,19827,<NA>
25,19827,L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL
27,19828,L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE
20,19828,<NA>


In [52]:
to_charleroi = "L C3-2: ERQUELINNES / MAUBEUGE -> CHARLEROI-CENTRAL"
from_charleroi = "L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES / MAUBEUGE"

lc32_to_charleroi_trains = (
    punctuality_lc32["TRAIN_NO"].loc[
        (punctuality_lc32["RELATION_DIRECTION"] == "L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL") |
        (punctuality_lc32["RELATION_DIRECTION"] == "L C3-2: MAUBEUGE -> CHARLEROI-CENTRAL")]
)

lc32_from_charleroi_trains = (
    punctuality_lc32["TRAIN_NO"].loc[
        (punctuality_lc32["RELATION_DIRECTION"] == "L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES") |
        (punctuality_lc32["RELATION_DIRECTION"] == "L C3-2: CHARLEROI-CENTRAL -> MAUBEUGE")]
)

mask_lc32_to_charleroi = (
    (punctuality["RELATION"] == "L C3-2") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lc32_to_charleroi_trains))
)

mask_lc32_from_charleroi = (
    (punctuality["RELATION"] == "L C3-2") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lc32_from_charleroi_trains))
)

punctuality.loc[mask_lc32_to_charleroi, "RELATION_DIRECTION"] = to_charleroi
punctuality.loc[mask_lc32_from_charleroi, "RELATION_DIRECTION"] = from_charleroi

In [53]:
# Verification
punctuality.loc[(punctuality["RELATION"] == "L C3-2") & (punctuality["RELATION_DIRECTION"].isnull())]

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [54]:
(
    punctuality[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "L C3-2"]
    .drop_duplicates().reset_index(drop=True)
).head(10)

,TRAIN_NO,RELATION,RELATION_DIRECTION
0,19836,L C3-2,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
1,19828,L C3-2,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
2,19835,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
3,19825,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
4,19832,L C3-2,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
5,19827,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
6,19833,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
7,19837,L C3-2,L C3-2: ERQUELINNES -> CHARLEROI-CENTRAL
8,19834,L C3-2,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES
9,19842,L C3-2,L C3-2: CHARLEROI-CENTRAL -> ERQUELINNES


### 5.1.3. Handling Missing Values in `RELATION_DIRECTION` for **L C2-2** `RELATION`

### 5.1.3.1. Profiling Missing Directions in `L C2-2` Relation

As established in Section 5.1.:
* `L C2-2` was an active train service for **98 days** only, from 6 September 2025 until 13 December 2025.  
* Its missing direction values account for **13.8%** of its `RELATION_DIRECTION` values.

As shown below:
* `L C2-2` serves **16 stations** between Charleroi and La Louvière, and has **16 train numbers**.  

* It has only two non-null unique `RELATION_DIRECTION` values:  
    * `L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE`
    * `L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL`

* Therefore, it has only two terminuses: `CHARLEROI-CENTRAL` and `LA LOUVIERE-CENTRE`.

In [55]:
train_service_lc22 = ["L C2-2"]

punctuality_lc22 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(train_service_lc22)]

print(f"L C2-2 total entries: {punctuality_lc22.shape[0]}")
print(f"The L C2-2 relation entries "
      f"represent {round(punctuality_lc22.shape[0] / punctuality.shape[0] * 100, 3)} % "
      f"entries of the punctuality dataset.")
print(" ")
print(f"TRAIN_NO: {len(punctuality_lc22["TRAIN_NO"].dropna().drop_duplicates())}")
print(f"RELATION_DIRECTION: {len(punctuality_lc22["RELATION_DIRECTION"].dropna().drop_duplicates())}")
print(f"Number of Stations: {len(punctuality_lc22["PTCAR_LG_NM_NL"].dropna().drop_duplicates())}")

L C2-2 total entries: 34746
The L C2-2 relation entries represent 0.076 % entries of the punctuality dataset.
 
TRAIN_NO: 16
RELATION_DIRECTION: 2
Number of Stations: 16


In [56]:
train_no_lc22 = punctuality_lc22["TRAIN_NO"].drop_duplicates().to_list()

In [57]:
station_names_lc22 = punctuality_lc22["PTCAR_LG_NM_NL"].drop_duplicates().to_list()

In [58]:
punctuality_lc22[["RELATION_DIRECTION"]].drop_duplicates().reset_index(drop=True)

,RELATION_DIRECTION
0,L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
1,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
2,<NA>


In [59]:
train_no_by_relation_direction_lc22 = (
    punctuality_lc22[["TRAIN_NO", "RELATION_DIRECTION"]].drop_duplicates().sort_values("TRAIN_NO")
    .reset_index(drop=True)
)

display(train_no_by_relation_direction_lc22.head())
train_no_by_relation_direction_lc22.tail()

,TRAIN_NO,RELATION_DIRECTION
0,4358,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
1,4358,<NA>
2,4360,<NA>
3,4360,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
4,4362,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL


,TRAIN_NO,RELATION_DIRECTION
27,4386,<NA>
28,4388,<NA>
29,4388,L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
30,4390,L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
31,4390,<NA>


As observed with other train services, the same `TRAIN_NO` unique value is always assigned to the same `RELATION_DIRECTION` value when this direction value is not missing.

In [60]:
alternative_relations_train_no_lc22 = (
    punctuality_to_check[["RELATION", "RELATION_DIRECTION"]]
    .loc[punctuality_to_check["TRAIN_NO"].isin(train_no_lc22)]
    .drop_duplicates().reset_index(drop=True)
)

alternative_relations_train_no_lc22.head(10)

,RELATION,RELATION_DIRECTION
0,L C2-2,L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
1,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
2,L 19,L 19: BRAINE-LE-COMTE -> LA LOUVIERE-SUD
3,L 19,L 19: LA LOUVIERE-SUD -> BRAINE-LE-COMTE
4,L C2-2,<NA>
5,L 19,<NA>
6,S62-2,S62-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
7,S62-2,S62-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL


* `L C2-2`, `S62-2`, and `L 19` relations share unique `TRAIN_NO` values.

In [61]:
lc22_s622_l19 = ["L C2-2", "S62-2", "L 19"]

punctuality_lc22_s622_l19 = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(lc22_s622_l19)]
)

alternative_relation_dates_lc22 = (
    punctuality_lc22_s622_l19.groupby("RELATION").agg(**AGG_RELATION_ACTIVITY).sort_values("relation_start")
)

alternative_relation_dates_lc22["relation_duration"] = (
    alternative_relation_dates_lc22["relation_end"] - alternative_relation_dates_lc22["relation_start"]
    )

alternative_relation_missing_spreads_lc22 = (
    punctuality_lc22_s622_l19.groupby("RELATION").agg(**AGG_MISSING_STATS).sort_values("missing_start")
)

alternative_relation_missing_spreads_lc22["missing_spread"] = (
    alternative_relation_missing_spreads_lc22["missing_end"] 
    - alternative_relation_missing_spreads_lc22["missing_start"]
)

print("----------------ALTERNATIVE LC 22 RELATIONS DATES--------------")
display(alternative_relation_dates_lc22)
print(" ")
print("---------ALTERNATIVE LC 22 MISSING DIRECTIONS SPREAD-----------")
alternative_relation_missing_spreads_lc22

----------------ALTERNATIVE LC 22 RELATIONS DATES--------------


,total_entries,missing_values,relation_start,relation_end,relation_duration
RELATION,,,,,
L C2-2,34746,4795,2024-01-01,2025-12-13,712 days
L 19,96115,176,2024-01-02,2025-12-31,729 days
S62-2,1056,0,2025-12-14,2025-12-28,14 days


 
---------ALTERNATIVE LC 22 MISSING DIRECTIONS SPREAD-----------


,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
L C2-2,34746,4795,2025-09-06,2025-12-13,98 days
L 19,96115,176,2025-11-11,2025-11-11,0 days
S62-2,1056,0,NaT,NaT,NaT


* As shown in the output above, the starting and ending dates of `L 19` and `L C2-2` do not match. Both train services were active at the same time. Therefore, one cannot be a relabeling of the other. 

* Furthermore, the only missing directions for the `L 19` `RELATION` class are concentrated on **a single day** - 11 November 2025. This appears to be an isolated incident unrelated to the missing `L C2-2` directions, which are spreading across **98 days**.

* By contrast, the `S62-2` train service starts on 14 December 2025, while `L C2-2` ends on 13 December 2025. `S62-2` do not have any missing directions. It is therefore most likely the new name of `L C2-2`.

In [62]:
station_names_s622 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality["RELATION"] == "S62-2"].drop_duplicates().to_list()
)

In [63]:
remaining_stations_lc22_s62 = list(set(station_names_lc22) - set(station_names_s622))
remaining_stations_lc22_s62

['LA LOUVIERE-SUD',
 'CHARLEROI-FAISCEAU A',
 'MONCEAU-RECEPTION',
 'MONCEAU-FORMATION',
 'MARCINELLE']

In [64]:
stations[["longnamefrench", "class_en"]].loc[stations["longnamefrench"].isin(remaining_stations_lc22_s62)]

,longnamefrench,class_en
245,LA LOUVIERE-SUD,Station
300,MONCEAU-RECEPTION,Station
566,CHARLEROI-FAISCEAU A,Station
722,MARCINELLE,Station
740,MONCEAU-FORMATION,Station


In [65]:
lc22_lc32 = ["L C2-2", "L C3-2"]

hypothesis_engineering_works_lc22 = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(lc22_lc32)]
)

missing_spreads_lc22_lc32 = (
    hypothesis_engineering_works_lc22.groupby("RELATION").agg(**AGG_MISSING_STATS)
)

missing_spreads_lc22_lc32["missing_spread"] = (
    missing_spreads_lc22_lc32["missing_end"] - missing_spreads_lc22_lc32["missing_start"]
)

missing_spreads_lc22_lc32

,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
L C2-2,34746,4795,2025-09-06,2025-12-13,98 days
L C3-2,97546,6912,2024-02-17,2025-02-02,351 days


As shown in the section above:

* `S62-2` starts on 14 December 2025, namely the day after `L C2-2` ends. As noted previously in this notebook, this is also the starting date of the third phase of the *SNCB Transport Plan 2023-2026*. 

* Like `S63` for `L C3-2`, `S62-2` serves more stations than `L C2-2` - `MONCEAU-FORMATION`, `MARCINELLE`, `MONCEAU-RECEPTION`, and `CHARLEROI-FAISCEAU A`, which are all within the **Charleroi network hub** - and `LA LOUVIERE-SUD`, which is one additional station before the terminus in **La Louvière**. 

* As a result, `S62-2` can be considered as both the new name and the extension of the `L C2-2` train service, serving the same line, but with five additional stops.

* However, the missing directions for `L C2-2` start on 6 September 2025 and end the day before `L C2-2` is relabeled as `S62-2`. They spread across **98 days**, which seems unusually long if they result from a discrepancy linked to the relabeling of `L C2-2`. 

* Nor do their starting and ending dates in 2025 align with those of the missing directions on `L C3-2` in 2024, which also serves the **Charleroi network hub**. Engineering works on the same infrastructure line should generate missing direction values within the same time period, which is not the case.

* Consequently, the origin of these missing directions remain unclear.

* Nevertheless, since `L C2-2` only has two `RELATION_DIRECTION` values and unique `TRAIN_NO` values are always assigned to the same `RELATION_DIRECTION` values, we can easily infer the correct `RELATION_DIRECTION` from the `TRAIN_NO` values to replace the missing directions, as we will do in Section 5.1.3.2. below.

### 5.1.3.2. Inferring Missing Directions for `L C2-2`

* As a reminder, the output below shows the current state of the `L C2-2 `RELATION_DIRECTION` values.

In [66]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "L C2-2"]
    .drop_duplicates().reset_index(drop=True)
)

,RELATION,RELATION_DIRECTION
0,L C2-2,L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE
1,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
2,L C2-2,<NA>


In [67]:
to_lalouviere = "L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE"
from_lalouviere = "L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL"

lc22_to_lalouviere_trains = (
    punctuality_lc22["TRAIN_NO"].loc[
        (punctuality_lc22["RELATION_DIRECTION"] == "L C2-2: CHARLEROI-CENTRAL -> LA LOUVIERE-CENTRE")
    ]
)

lc22_from_lalouviere_trains = (
    punctuality_lc22["TRAIN_NO"].loc[
        (punctuality_lc22["RELATION_DIRECTION"] == "L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL")
        ]
)

mask_lc22_to_lalouviere = (
    (punctuality["RELATION"] == "L C2-2") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lc22_to_lalouviere_trains))
)

mask_lc22_from_lalouviere = (
    (punctuality["RELATION"] == "L C2-2") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lc22_from_lalouviere_trains))
)

punctuality.loc[mask_lc22_to_lalouviere, "RELATION_DIRECTION"] = to_lalouviere
punctuality.loc[mask_lc22_from_lalouviere, "RELATION_DIRECTION"] = from_lalouviere

In [68]:
# Verification
punctuality.loc[(punctuality["RELATION"] == "L C2-2") & (punctuality["RELATION_DIRECTION"].isnull())]

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [69]:
(
    punctuality[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[(punctuality["RELATION"] == "L C2-2") &
         (punctuality["TRAIN_NO"].isin(train_no_lc22))]
         .drop_duplicates().reset_index(drop=True).sort_values("TRAIN_NO")
).head()

,TRAIN_NO,RELATION,RELATION_DIRECTION
3,4358,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
4,4360,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
5,4362,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
2,4364,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL
7,4366,L C2-2,L C2-2: LA LOUVIERE-CENTRE -> CHARLEROI-CENTRAL


### 5.1.4. Handling Missing Values in `RELATION_DIRECTION` for `L B5-1` `RELATION`

### 5.1.4.1. Profiling Missing Directions for `L B5-1` Relation

As established in Section 5.1.:  
* `L B5-1` was active for **713 days** during the 2024-2025 period. It ceased operation on **13 December 2025**.

* Its missing direction rate is **~2.4%**.

As shown below:  
* `L B5-1` entries account for **0.98 %** of `punctuality`.  

* `L B5-1` is a major train service that operates with **38 train numbers** and serves **89 stations and train stops** within or near the **Brussels network hub**.

* `L B5-1` has four unique non-null `RELATION_DIRECTION` values:
    * `L B5-1: MECHELEN -> HALLE`
    * `L B5-1: HALLE -> MECHELEN`
    * `L B5-1: EDINGEN -> MECHELEN`
    * `L B5-1: MECHELEN -> EDINGEN`

* Therefore, it has three terminuses: `HALLE` and `EDINGEN` downstream, and `MECHELEN` upstream.

In [70]:
train_service_lb51 = ["L B5-1"]

punctuality_lb51 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(train_service_lb51)]

print(f"L B5-1 total entries: {punctuality_lb51.shape[0]}")
print(f"The L B5-1 entries represent {round(punctuality_lb51.shape[0] / punctuality.shape[0] * 100, 2)}% "
      f"of the total entries of the punctuality dataset.")
print(" ")
print(f"TRAIN_NO: {len(punctuality_lb51["TRAIN_NO"].dropna().drop_duplicates())}")
print(f"RELATION_DIRECTION: {len(punctuality_lb51["RELATION_DIRECTION"].dropna().drop_duplicates())}")
print(f"Number of Stations: {len(punctuality_lb51["PTCAR_LG_NM_NL"].dropna().drop_duplicates())}")

L B5-1 total entries: 446585
The L B5-1 entries represent 0.98% of the total entries of the punctuality dataset.
 
TRAIN_NO: 38
RELATION_DIRECTION: 4
Number of Stations: 89


In [71]:
punctuality_lb51[["RELATION", "RELATION_DIRECTION"]].drop_duplicates().reset_index(drop=True)

,RELATION,RELATION_DIRECTION
0,L B5-1,L B5-1: MECHELEN -> HALLE
1,L B5-1,L B5-1: HALLE -> MECHELEN
2,L B5-1,<NA>
3,L B5-1,L B5-1: EDINGEN -> MECHELEN
4,L B5-1,L B5-1: MECHELEN -> EDINGEN


In [72]:
lb51_without_direction = (
    punctuality_lb51[["RELATION", "RELATION_DIRECTION", "PTCAR_LG_NM_NL", "TRAIN_NO", "DATDEP"]]
    .loc[punctuality_lb51["RELATION_DIRECTION"].isnull()]
)
stations_without_direction_lb51 = (
    lb51_without_direction["PTCAR_LG_NM_NL"].drop_duplicates().reset_index(drop=True).to_list()
)
print(f"From {len(punctuality_lb51["PTCAR_LG_NM_NL"].dropna().drop_duplicates())} stations and stops, "
      f"{len(stations_without_direction_lb51)} are served by L B5-1 relation with no direction.")

train_no_without_direction_lb51 = (
    lb51_without_direction["TRAIN_NO"].drop_duplicates().reset_index(drop=True).to_list()
)

print(f"{len(train_no_without_direction_lb51)} TRAIN_NO values are associated to missing L B5-1 directions.")

From 89 stations and stops, 43 are served by L B5-1 relation with no direction.
36 TRAIN_NO values are associated to missing L B5-1 directions.


In [73]:
train_no_lb51 = punctuality_lb51["TRAIN_NO"].drop_duplicates().to_list()

In [74]:
stations_lb51 = punctuality_lb51["PTCAR_LG_NM_NL"].drop_duplicates().to_list()

In [75]:
alternative_relations_lb51 = punctuality_to_check.loc[punctuality_to_check["TRAIN_NO"].isin(train_no_lb51)]

alternative_relations_lb51["RELATION"].drop_duplicates().reset_index(drop=True)

0    L B5-1
1      S5-1
Name: RELATION, dtype: string

In [76]:
stations_s51 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality["RELATION"] == "S5-1"]
    .drop_duplicates().to_list()
)

In [77]:
print(stations_lb51 == stations_s51)

False


In [78]:
remaining_stations_lb51 = (set(stations_lb51) - set(stations_s51))
print(f"S5-1 serves {len(stations_s51)} stations.")
print(f"L B5-1 has {len(remaining_stations_lb51)} more stations than S5-1.")

S5-1 serves 32 stations.
L B5-1 has 57 more stations than S5-1.


In [79]:
lb51_s51 = ["L B5-1", "S5-1"]

punctuality_lb51_s51 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(lb51_s51)]

relation_dates_lb51_s51 = (
    punctuality_lb51_s51.groupby("RELATION").agg(**AGG_RELATION_ACTIVITY).sort_values("relation_start")
)

relation_dates_lb51_s51["relation_duration"] = (
    relation_dates_lb51_s51["relation_end"] - relation_dates_lb51_s51["relation_start"]
    )

missing_stats_lb51 = (
    punctuality_lb51_s51.groupby("RELATION").agg(**AGG_MISSING_STATS).sort_values("missing_start")
)

missing_stats_lb51["missing_spread"] = (
    missing_stats_lb51["missing_end"] - missing_stats_lb51["missing_start"]
)

print("---------------------ALTERNATIVE L B5-1 RELATIONS DATES-------------------")
display(relation_dates_lb51_s51)
print(" ")
print("-----------------------L B5-1 MISSING DIRECTIONS SPREAD-------------------")
missing_stats_lb51

---------------------ALTERNATIVE L B5-1 RELATIONS DATES-------------------


,total_entries,missing_values,relation_start,relation_end,relation_duration
RELATION,,,,,
L B5-1,446585,10818,2024-01-01,2025-12-13,712 days
S5-1,13752,0,2025-12-14,2025-12-31,17 days


 
-----------------------L B5-1 MISSING DIRECTIONS SPREAD-------------------


,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
L B5-1,446585,10818,2024-01-02,2024-04-12,101 days
S5-1,13752,0,NaT,NaT,NaT


As shown in the section above:  

* Although the unique `TRAIN_NO` values serving the `L B5-1` train service are also assigned to `S5-1`, and `S5-1` starts precisely one day after `L B5-1` ceased operating - **14 December 2025**, **they are not the same train service**: 
    * `S5-1` serves **32** stations whereas `L B5-1` served **89** stations. The two relations do not serve the same stations but have only **32 stations in common**. As a result, `S5-1` appears to have replaced a part of the former `L B5-1` train service.

* The missing `RELATION_DIRECTION` values for `L B5-1` are all located in **2024**. Therefore, the renaming of `L B5-1` in 2025 cannot have explained its missing direction values anyway.  

* Furthermore, these missing values concern only part of the entries of the `L B5-1` relation for these 2024 **101 days**. Other `L B5-1` entries have valid directions during the same period. This may support the *Brussels RER* engineering works hypothesis presented at the beginning of this section. 

* Yet, the cause of the `L B5-1` missing direction values remains unclear.

Nevertheless, as shown in the output below:

* The **general orientation** of `L B5-1` , **from MECHELEN** or **to MECHELEN**, is consistent across specific train numbers, even though the exact terminus varies. It can be either **EDINGEN** or **HALLE**.

* Therefore, the `L B5-1` missing directions are replaced by **one of the two following values**:
    * `L B5-1: MECHELEN -> HALLE / EDINGEN`
    * `L B5-1: EDINGEN / HALLE -> MECHELEN`

In [80]:
lb51 = ["L B5-1"]

train_no_missing_direction_lb51 = (
    punctuality_to_check[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[(punctuality_to_check["TRAIN_NO"].isin(train_no_lb51))
         & punctuality_to_check["RELATION"].isin(lb51)]
    .drop_duplicates().sort_values("TRAIN_NO").reset_index(drop=True)
    )

display(train_no_missing_direction_lb51.head(10))
train_no_missing_direction_lb51.tail(10)


,TRAIN_NO,RELATION,RELATION_DIRECTION
0,3354,L B5-1,L B5-1: EDINGEN -> MECHELEN
1,3355,L B5-1,<NA>
2,3355,L B5-1,L B5-1: EDINGEN -> MECHELEN
3,3356,L B5-1,<NA>
4,3356,L B5-1,L B5-1: EDINGEN -> MECHELEN
5,3357,L B5-1,L B5-1: EDINGEN -> MECHELEN
6,3357,L B5-1,L B5-1: HALLE -> MECHELEN
7,3357,L B5-1,<NA>
8,3358,L B5-1,<NA>
9,3358,L B5-1,L B5-1: HALLE -> MECHELEN


,TRAIN_NO,RELATION,RELATION_DIRECTION
92,3390,L B5-1,L B5-1: MECHELEN -> HALLE
93,3391,L B5-1,L B5-1: MECHELEN -> HALLE
94,3391,L B5-1,L B5-1: MECHELEN -> EDINGEN
95,3391,L B5-1,<NA>
96,3392,L B5-1,L B5-1: MECHELEN -> HALLE
97,3392,L B5-1,<NA>
98,3392,L B5-1,L B5-1: MECHELEN -> EDINGEN
99,3393,L B5-1,<NA>
100,3393,L B5-1,L B5-1: MECHELEN -> EDINGEN
101,3394,L B5-1,L B5-1: MECHELEN -> EDINGEN


### 5.1.4.2. Inferring Directions for `L B5-1`

* As a reminder, the output below shows the current state of the `L B5-1` `RELATION_DIRECTION` values.

In [81]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "L B5-1"]
    .drop_duplicates().reset_index(drop=True)
)

,RELATION,RELATION_DIRECTION
0,L B5-1,L B5-1: MECHELEN -> HALLE
1,L B5-1,L B5-1: HALLE -> MECHELEN
2,L B5-1,<NA>
3,L B5-1,L B5-1: EDINGEN -> MECHELEN
4,L B5-1,L B5-1: MECHELEN -> EDINGEN


In [82]:
to_mechelen = "L B5-1: EDINGEN / HALLE -> MECHELEN"
from_mechelen = "L B5-1: MECHELEN -> HALLE / EDINGEN"

lb51_to_mechelen_trains = (
    punctuality_lb51["TRAIN_NO"].loc[
        (punctuality_lb51["RELATION_DIRECTION"] == "L B5-1: HALLE -> MECHELEN") |
        (punctuality_lb51["RELATION_DIRECTION"] == "L B5-1: EDINGEN -> MECHELEN")]
)

lb51_from_mechelen_trains = (
    punctuality_lb51["TRAIN_NO"].loc[
        (punctuality_lb51["RELATION_DIRECTION"] == "L B5-1: MECHELEN -> HALLE") |
        (punctuality_lb51["RELATION_DIRECTION"] == "L B5-1: MECHELEN -> EDINGEN")]
)

mask_lb51_to_mechelen = (
    (punctuality["RELATION"] == "L B5-1") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lb51_to_mechelen_trains))
)

mask_lb51_from_mechelen = (
    (punctuality["RELATION"] == "L B5-1") &
    (punctuality["RELATION_DIRECTION"].isnull()) &
    (punctuality["TRAIN_NO"].isin(lb51_from_mechelen_trains))
)


punctuality.loc[mask_lb51_to_mechelen, "RELATION_DIRECTION"] = to_mechelen
punctuality.loc[mask_lb51_from_mechelen, "RELATION_DIRECTION"] = from_mechelen

In [83]:
# Verification
punctuality.loc[(punctuality["RELATION"] == "L B5-1") & (punctuality["RELATION_DIRECTION"].isnull())]

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [84]:
verification_lb51 = (
    punctuality[["TRAIN_NO", "RELATION", "RELATION_DIRECTION"]]
    .loc[punctuality["TRAIN_NO"].isin(train_no_lb51) &
         punctuality["RELATION"].isin(["L B5-1"])]
    .drop_duplicates().sort_values("TRAIN_NO").reset_index(drop=True)
    )

display(verification_lb51.head())
verification_lb51.tail()

,TRAIN_NO,RELATION,RELATION_DIRECTION
0,3354,L B5-1,L B5-1: EDINGEN -> MECHELEN
1,3355,L B5-1,L B5-1: EDINGEN / HALLE -> MECHELEN
2,3355,L B5-1,L B5-1: EDINGEN -> MECHELEN
3,3356,L B5-1,L B5-1: EDINGEN / HALLE -> MECHELEN
4,3356,L B5-1,L B5-1: EDINGEN -> MECHELEN


,TRAIN_NO,RELATION,RELATION_DIRECTION
97,3392,L B5-1,L B5-1: MECHELEN -> HALLE / EDINGEN
98,3392,L B5-1,L B5-1: MECHELEN -> EDINGEN
99,3393,L B5-1,L B5-1: MECHELEN -> HALLE / EDINGEN
100,3393,L B5-1,L B5-1: MECHELEN -> EDINGEN
101,3394,L B5-1,L B5-1: MECHELEN -> EDINGEN


### 5.1.5. Handling Missing Values in `RELATION_DIRECTION` for `IC 20` `RELATION`

### 5.1.5.1. Profiling Missing Directions for `IC 20` Relation

As established in Section 5.1.:
* `IC 20` spans the entire dataset period (2024-2025). Therefore, it was not renamed during this period.

* Its missing direction rate is **6.75%**.

As shown below:  
* `IC 20` is an extensive train service that serves **144 stations and train stops**, and operates with **78 train numbers**.  

* Above all, this relation has **6** distinct unique `RELATION_DIRECTION` values:  
    * `IC 20: GENT-SINT-PIETERS -> LOKEREN`  
    * `IC 20: GENT-SINT-PIETERS -> TONGEREN`  
    * `IC 20: KNOKKE -> LIEGE-GUILLEMINS`  
    * `IC 20: LOKEREN -> GENT-SINT-PIETERS`  
    * `IC 20: TONGEREN -> GENT-SINT-PIETERS`  
    * `IC 20: LIEGE-GUILLEMINS -> KNOKKE`

* Therefore, this relation has five possible terminuses: `TONGEREN` and `LIEGE-GUILLEMINS` to the east, and `GENT-SINT-PIETERS`, `LOKEREN`, and `KNOKKE` to the west.  

* Combined with the high number of train numbers, these six distinct direction values and five terminuses significantly complicates replacing the missing values by the correct directions.

In [85]:
train_service_ic20 = ["IC 20"]

punctuality_ic20 = punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(train_service_ic20)]

print(f"IC 20 total entries: {punctuality_ic20.shape[0]}")
print(f"The IC 20 entries represent {round(punctuality_ic20.shape[0] / punctuality.shape[0] * 100, 2)}% "
      f"of the total entries of the punctuality dataset.")
print(f" ")
print(f"TRAIN_NO: {len(punctuality_ic20["TRAIN_NO"].dropna().drop_duplicates())}")
print(f"RELATION_DIRECTION: {len(punctuality_ic20["RELATION_DIRECTION"].dropna().drop_duplicates())}")
print(f"Number of Stations: {len(punctuality_ic20["PTCAR_LG_NM_NL"].dropna().drop_duplicates())}")

IC 20 total entries: 821563
The IC 20 entries represent 1.8% of the total entries of the punctuality dataset.
 
TRAIN_NO: 78
RELATION_DIRECTION: 6
Number of Stations: 144


In [86]:
punctuality_ic20[["RELATION", "RELATION_DIRECTION"]].drop_duplicates().reset_index(drop=True)

,RELATION,RELATION_DIRECTION
0,IC 20,IC 20: LOKEREN -> GENT-SINT-PIETERS
1,IC 20,IC 20: GENT-SINT-PIETERS -> LOKEREN
2,IC 20,IC 20: GENT-SINT-PIETERS -> TONGEREN
3,IC 20,IC 20: TONGEREN -> GENT-SINT-PIETERS
4,IC 20,<NA>
5,IC 20,IC 20: KNOKKE -> LIEGE-GUILLEMINS
6,IC 20,IC 20: LIEGE-GUILLEMINS -> KNOKKE


In [87]:
train_no_ic20 = punctuality_ic20["TRAIN_NO"].drop_duplicates().to_list()

In [88]:
station_names_ic20 = punctuality_ic20["PTCAR_LG_NM_NL"].drop_duplicates().to_list()

In [89]:
alternative_relations_to_ic20 = (
    punctuality_to_check.loc[punctuality_to_check["TRAIN_NO"].isin(train_no_ic20)]
)
alternative_relations_to_ic20["RELATION"].drop_duplicates().reset_index(drop=True)

0    IC 23-2
1      IC 20
2    IC 20-1
3    IC 23-1
Name: RELATION, dtype: string

In [90]:
stations_names_ic232 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality_to_check["RELATION"] == "IC 23-2"]
    .drop_duplicates().to_list()
)

In [91]:
stations_names_ic231 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality_to_check["RELATION"] == "IC 23-1"]
    .drop_duplicates().to_list()
)

In [92]:
stations_names_ic201 = (
    punctuality_to_check["PTCAR_LG_NM_NL"].loc[punctuality_to_check["RELATION"] == "IC 20-1"]
    .drop_duplicates().to_list()
)

In [93]:
print(len(stations_names_ic201))
print(len(stations_names_ic231))
print(len(stations_names_ic232))
print(len(station_names_ic20))

83
82
82
144


The three train services served by the same unique `TRAIN_NO` values than `IC 20` have more than **60 stations less**. There are not the same train service.

In [94]:
ic20_ic201_ic231_ic232 = ["IC 20", "IC 20-1", "IC 23-1", "IC 23-2"]

punctuality_possible_alternative_relations_ic20 = (
    punctuality_to_check.loc[punctuality_to_check["RELATION"].isin(ic20_ic201_ic231_ic232)]
)

possible_alternative_relation_stats_ic20 = (
    punctuality_possible_alternative_relations_ic20    
    .groupby("RELATION").agg(**AGG_RELATION_ACTIVITY).sort_values("relation_start")
)

possible_alternative_relation_stats_ic20["relation_active_days"] = (
    (possible_alternative_relation_stats_ic20["relation_end"]) - 
    (possible_alternative_relation_stats_ic20["relation_start"])
)

possible_alternative_relation_missing_spread_ic20 = (
    punctuality_possible_alternative_relations_ic20
    .groupby("RELATION").agg(**AGG_MISSING_STATS)
)

possible_alternative_relation_missing_spread_ic20["missing_spread"] = (
    (possible_alternative_relation_missing_spread_ic20["missing_end"]) - 
    (possible_alternative_relation_missing_spread_ic20["missing_start"])
)

print("-----------------------ALTERNATIVE IC 20 RELATIONS DATES---------------------")
display(possible_alternative_relation_stats_ic20)
print(" ")
print("----------------ALTERNATIVE IC 20 MISSING DIRECTIONS SPREAD------------------")
possible_alternative_relation_missing_spread_ic20

-----------------------ALTERNATIVE IC 20 RELATIONS DATES---------------------


,total_entries,missing_values,relation_start,relation_end,relation_active_days
RELATION,,,,,
IC 20,821563,55423,2024-01-01,2025-12-31,730 days
IC 23-1,597312,0,2024-01-01,2025-12-31,730 days
IC 23-2,655346,0,2024-01-01,2025-12-31,730 days
IC 20-1,75104,0,2024-09-02,2024-10-31,59 days


 
----------------ALTERNATIVE IC 20 MISSING DIRECTIONS SPREAD------------------


,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
IC 20,821563,55423,2024-11-01,2024-12-14,43 days
IC 20-1,75104,0,NaT,NaT,NaT
IC 23-1,597312,0,NaT,NaT,NaT
IC 23-2,655346,0,NaT,NaT,NaT


* The output above shows clearly that `IC 23-1` and `IC 23-2` were not alternative names for `IC 20`, even though they share unique train numbers.

* However, the missing directions in the `IC 20` train service start on **1 November 2024**, while the `IC 20-1` train service ends precisely on **31 October 2024**. A plausible hypothesis is that `IC 20` was temporarily labeled `IC 20-1` from **2 September 2024** to **31 October 2024**, and that the correct `IC 20` `RELATION_DIRECTION` values directions had not yet been implemented when the relation was reassigned as `IC 20` again. Alternatively, this may indicate a desynchronization between real-time tracking (*ARAMIS system*) and platform route planing (*OP1 platform*). This may support the *Brussels RER* engineering works hypothesis presented at the beginning of this section.

* Nevertheless, this information is not sufficient to infer the correct missing `RELATION_DIRECTION` values for `IC 20`. 

### 5.1.5.2. Inferring Directions for `IC 20`

* The missing directions for `IC 20` are replaced by *" - Direction Unknown (Works Impact)"*  

* As a reminder, the output below shows the current state of its `RELATION_DIRECTION` values.

In [95]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "IC 20"]
    .drop_duplicates().reset_index(drop=True)
)

,RELATION,RELATION_DIRECTION
0,IC 20,IC 20: LOKEREN -> GENT-SINT-PIETERS
1,IC 20,IC 20: GENT-SINT-PIETERS -> LOKEREN
2,IC 20,IC 20: GENT-SINT-PIETERS -> TONGEREN
3,IC 20,IC 20: TONGEREN -> GENT-SINT-PIETERS
4,IC 20,<NA>
5,IC 20,IC 20: KNOKKE -> LIEGE-GUILLEMINS
6,IC 20,IC 20: LIEGE-GUILLEMINS -> KNOKKE


In [96]:
mask = punctuality["RELATION"].isin(["IC 20"]) & punctuality["RELATION_DIRECTION"].isnull()

punctuality.loc[mask, "RELATION_DIRECTION"] = "IC 20 - Unknown Direction - Works Impact"

In [97]:
# Verification
punctuality.loc[(punctuality["RELATION"] == "IC 20") & (punctuality["RELATION_DIRECTION"].isnull())]

,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE


In [98]:
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"] == "IC 20"]
    .drop_duplicates().reset_index(drop=True)
)

,RELATION,RELATION_DIRECTION
0,IC 20,IC 20: LOKEREN -> GENT-SINT-PIETERS
1,IC 20,IC 20: GENT-SINT-PIETERS -> LOKEREN
2,IC 20,IC 20: GENT-SINT-PIETERS -> TONGEREN
3,IC 20,IC 20: TONGEREN -> GENT-SINT-PIETERS
4,IC 20,IC 20 - Unknown Direction - Works Impact
5,IC 20,IC 20: KNOKKE -> LIEGE-GUILLEMINS
6,IC 20,IC 20: LIEGE-GUILLEMINS -> KNOKKE


### 5.1.6. Handling Missing values in `RELATION_DIRECTION` for Other `RELATION` Classes

As established in Section 5.1.:  

* The remaining `L 19`, `THAL`, `L 28`, `INT`, `L 12`, and `L 43` `RELATION` classes are active during almost the entire dataset period (**between 727 and 730 out of 730 days**).

* Their missing direction rates vary from **0.003%** to **0.89%** of their `RELATION_DIRECTION` values.

* `L 19`, `THAL`, and `L 28` have missing direction values concentrated on **a single day**.

In [99]:
last_relations_to_check = ["L 19", "THAL", "L 28", "INT", "L 12", "L 43"]

relations_with_missing_directions = (
    punctuality_to_check[punctuality_to_check["RELATION"].isin(last_relations_to_check)]
    .groupby("RELATION").agg(**AGG_RELATION_ACTIVITY)
)

relations_with_missing_directions["relation_duration"] = (
    relations_with_missing_directions["relation_end"] - relations_with_missing_directions["relation_start"]
)

relations_with_missing_directions["pct_of_dataset"] = (
    relations_with_missing_directions["total_entries"] / punctuality.shape[0] * 100
    ).round(3)

missing_direction_stats = (
    punctuality_to_check[punctuality_to_check["RELATION"].isin(last_relations_to_check)]
    .groupby("RELATION").agg(**AGG_MISSING_STATS)
)

missing_direction_stats["missing_spread"] = (
    missing_direction_stats["missing_end"] - missing_direction_stats["missing_start"]
)

print("--------------------ACTIVITY PERIOD AND WEIGHT OF REMAINING RELATIONS ----------------------")
display(relations_with_missing_directions.sort_values("pct_of_dataset", ascending=False))
print(" ")
print("------------SPREAD OF MISSING DIRECTIONS IN REMAINING RELATIONS--------------")
missing_direction_stats.sort_values("missing_spread", ascending=False)


--------------------ACTIVITY PERIOD AND WEIGHT OF REMAINING RELATIONS ----------------------


,total_entries,missing_values,relation_start,relation_end,relation_duration,pct_of_dataset
RELATION,,,,,,
INT,271493,1455,2024-01-01,2025-12-31,730 days,0.596
THAL,258694,20,2024-01-01,2025-12-21,720 days,0.568
L 19,96115,176,2024-01-02,2025-12-31,729 days,0.211
L 28,91835,3,2024-01-01,2025-12-28,727 days,0.202
L 12,17433,92,2024-01-02,2025-12-31,729 days,0.038
L 43,15216,135,2024-01-01,2025-12-31,730 days,0.033


 
------------SPREAD OF MISSING DIRECTIONS IN REMAINING RELATIONS--------------


,total_entries,missing_values,missing_start,missing_end,missing_spread
RELATION,,,,,
INT,271493,1455,2024-04-08,2025-12-28,629 days
L 43,15216,135,2024-01-01,2025-02-02,398 days
L 12,17433,92,2024-11-11,2025-11-11,365 days
L 19,96115,176,2025-11-11,2025-11-11,0 days
L 28,91835,3,2024-09-16,2024-09-16,0 days
THAL,258694,20,2025-12-21,2025-12-21,0 days


In [100]:
print(f"The missing directions of these six remaining relations account all together for "
      f"{round(relations_with_missing_directions["missing_values"].sum() / punctuality.shape[0] *100, 3)}% "
      f"of the punctuality dataset.")

The missing directions of these six remaining relations account all together for 0.004% of the punctuality dataset.


Since the remaining missing directions account for less than **0.01%** of `punctuality` entries, they will not have any significant statistical impact on the further analysis. Therefore, for each of them, we replace the missing `RELATION_DIRECTION` values with the `RELATION` class value followed by the `str` `" - direction unknown"`, to ensure the `train_service_dimension` **integrity**.

In [101]:
punctuality["RELATION_DIRECTION"] = (
    punctuality["RELATION_DIRECTION"].fillna(punctuality["RELATION"] + " - Unknown Direction")
)

In [102]:
# Verification
(
    punctuality[["RELATION", "RELATION_DIRECTION"]].loc[punctuality["RELATION"].isin(last_relations_to_check)]
    .drop_duplicates().reset_index(drop=True)
 )

,RELATION,RELATION_DIRECTION
0,INT,INT: BRUSSEL-ZUID -> PRAHA HL.N.
1,L 43,L 43 - Unknown Direction
2,L 28,L 28: MECHELEN -> KORTRIJK
3,L 28,L 28: KORTRIJK -> MECHELEN
4,THAL,THAL: PARIS-NORD -> AMSTERDAM CENTRAAL
5,THAL,THAL: AMSTERDAM CENTRAAL -> PARIS-NORD
6,THAL,THAL: KOLN HBF -> PARIS-NORD
7,THAL,THAL: PARIS-NORD -> KOLN HBF
8,THAL,THAL: AMSTERDAM CENTRAAL -> MARNE-LA-VALLEE-CH...
9,THAL,THAL: MARNE-LA-VALLEE-CHESSY -> AMSTERDAM CENT...


## 5.2. Export to Silver Layer

The `punctuality` DataFrame is now ready to be exported to the **intermediate directory**, as `punctuality_cleaned.parquet`.

In [103]:
punctuality.to_parquet(PATH_INTERMEDIATE / "punctuality_cleaned.parquet")

In [104]:
del punctuality

In [105]:
df_to_verify = pd.read_parquet(PATH_INTERMEDIATE / "punctuality_cleaned.parquet")

In [106]:
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())
df_to_verify.head()

(45542084, 21) {'DATDEP': dtype('<M8[us]'), 'TRAIN_NO': Int64Dtype(), 'RELATION': <StringDtype(na_value=<NA>)>, 'TRAIN_SERV': <StringDtype(na_value=<NA>)>, 'PTCAR_NO': Int64Dtype(), 'LINE_NO_DEP': <StringDtype(na_value=<NA>)>, 'REAL_TIME_ARR': <StringDtype(na_value=<NA>)>, 'REAL_TIME_DEP': <StringDtype(na_value=<NA>)>, 'PLANNED_TIME_ARR': <StringDtype(na_value=<NA>)>, 'PLANNED_TIME_DEP': <StringDtype(na_value=<NA>)>, 'DELAY_ARR': dtype('float64'), 'DELAY_DEP': dtype('float64'), 'RELATION_DIRECTION': <StringDtype(na_value=<NA>)>, 'PTCAR_LG_NM_NL': <StringDtype(na_value=<NA>)>, 'LINE_NO_ARR': <StringDtype(na_value=<NA>)>, 'PLANNED_DATE_ARR': dtype('<M8[us]'), 'PLANNED_DATE_DEP': dtype('<M8[us]'), 'REAL_DATE_ARR': dtype('<M8[us]'), 'REAL_DATE_DEP': dtype('<M8[us]'), 'DIRECTION_IS_INFERRED': Int64Dtype(), 'RELATION_DIRECTION_MISSING_DATE': dtype('<M8[us]')}


,DATDEP,TRAIN_NO,RELATION,TRAIN_SERV,PTCAR_NO,LINE_NO_DEP,REAL_TIME_ARR,REAL_TIME_DEP,PLANNED_TIME_ARR,PLANNED_TIME_DEP,...,DELAY_DEP,RELATION_DIRECTION,PTCAR_LG_NM_NL,LINE_NO_ARR,PLANNED_DATE_ARR,PLANNED_DATE_DEP,REAL_DATE_ARR,REAL_DATE_DEP,DIRECTION_IS_INFERRED,RELATION_DIRECTION_MISSING_DATE
0,2024-01-01,1813,IC 02,SNCB/NMBS,929,50A,<NA>,13:09:42,<NA>,13:09:00,...,42.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTENDE,<NA>,NaT,2024-01-01,NaT,2024-01-01,0,NaT
1,2024-01-01,1813,IC 02,SNCB/NMBS,210,50A,13:22:17,13:25:20,13:22:00,13:25:00,...,20.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BRUGGE,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0,NaT
2,2024-01-01,1813,IC 02,SNCB/NMBS,931,50A,13:29:18,13:29:18,13:28:00,13:28:00,...,78.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,OOSTKAMP,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0,NaT
3,2024-01-01,1813,IC 02,SNCB/NMBS,127,50A,13:31:59,13:31:59,13:31:00,13:31:00,...,59.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,BEERNEM,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0,NaT
4,2024-01-01,1813,IC 02,SNCB/NMBS,797,50A,13:33:57,13:33:57,13:33:00,13:33:00,...,57.0,IC 02: OOSTENDE -> ANTWERPEN-CENTRAAL,MARIA-AALTER,50A,2024-01-01,2024-01-01,2024-01-01,2024-01-01,0,NaT


In [107]:
# Last verification
df_to_verify["RELATION_DIRECTION"].isnull().sum()

np.int64(0)